In [ ]:
# Import libraries
import yfinance as yf
import plotly.graph_objects as go
import plotly.offline as pyo

# Configure Plotly for Jupyter Notebook
pyo.init_notebook_mode(connected=True)

print("✓ Libraries loaded successfully")

In [ ]:
# Configuration - CHANGE THESE VALUES AS NEEDED
ticker = "AAPL"  # Stock symbol (try: MSFT, GOOGL, TSLA, SAP)
start_date = "2023-01-01"
end_date = "2024-11-15"

# Download data
print(f"Loading data for {ticker}...")
data = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Fix datetime display issue - remove timezone info
data.index = data.index.tz_localize(None)

print(f"✓ Data loaded: {len(data)} trading days")
print(f"Price range: ${float(data['Close'].min()):.2f} - ${float(data['Close'].max()):.2f}")

In [ ]:
# Create interactive line chart
fig = go.Figure()

# Add price line
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Close'],
    mode='lines',
    name='Close Price',
    line=dict(color='#2E86AB', width=2.5),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>'
))

# Layout
fig.update_layout(
    title={
        'text': f'{ticker} Stock Price - Find Support & Resistance Levels',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial'}  
    },
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    hovermode='x unified',
    template='plotly_white',
    height=600,
    xaxis=dict(
        rangeslider=dict(visible=False),
        showgrid=True,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    )
)

fig.show()

print("\n💡 TIP: You can zoom, pan, and hover over the chart!")
print("📝 Write down the support and resistance levels you identify")

In [ ]:
# ENTER YOUR IDENTIFIED LEVELS HERE:

# Support levels (where price bounced UP)
support_levels = [
    165,  # Example - replace with your values
    175,  # Add more as needed
]

# Resistance levels (where price bounced DOWN)
resistance_levels = [
    195,  # Example - replace with your values
    200,  # Add more as needed
]

print("Your identified levels:")
print(f"Support: {support_levels}")
print(f"Resistance: {resistance_levels}")

In [ ]:
# Create chart with your identified lines
fig = go.Figure()

# Add price line
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Close'],
    mode='lines',
    name='Close Price',
    line=dict(color='#2E86AB', width=2.5),
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:.2f}<extra></extra>'
))

# Add support lines
for level in support_levels:
    fig.add_hline(
        y=level,
        line_dash="dash",
        line_color="green",
        line_width=2.5,
        opacity=0.7,
        annotation_text=f"Support: ${level}",
        annotation_position="right",
        annotation=dict(
            font=dict(size=11, color="green"),
            bgcolor="white"
        )
    )

# Add resistance lines
for level in resistance_levels:
    fig.add_hline(
        y=level,
        line_dash="dash",
        line_color="red",
        line_width=2.5,
        opacity=0.7,
        annotation_text=f"Resistance: ${level}",
        annotation_position="right",
        annotation=dict(
            font=dict(size=11, color="red"),
            bgcolor="white"
        )
    )

# Layout
fig.update_layout(
    title={
        'text': f'{ticker} - Your Support & Resistance Analysis',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial'}
    },
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    hovermode='x unified',
    template='plotly_white',
    height=650,
    xaxis=dict(
        rangeslider=dict(visible=False),
        showgrid=True,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    )
)

fig.show()

print("\n✓ Your manually identified levels are now visualized!")

In [ ]:
def count_touches(data, level, tolerance_percent=2):
    """
    Count how many times price came close to a level
    
    Parameters:
    - data: DataFrame with High/Low prices
    - level: Support or resistance price level
    - tolerance_percent: How close (in %) counts as a touch
    
    Returns:
    - count: Number of touches
    - dates: List of dates when touches occurred
    """
    tolerance = level * (tolerance_percent / 100)
    lower = level - tolerance
    upper = level + tolerance
    
    # Price touched the level if it came within the tolerance zone
    touches = ((data['Low'] <= upper) & (data['High'] >= lower))
    touch_dates = data[touches].index.tolist()
    
    return len(touch_dates), touch_dates

# Validate Support levels
print("="*70)
print("SUPPORT LEVELS VALIDATION:")
print("="*70)
for level in support_levels:
    count, dates = count_touches(data, level, tolerance_percent=2)
    strength = "STRONG 💪" if count >= 3 else "MODERATE ⚠️" if count == 2 else "WEAK ❓"
    print(f"\n${level}: {count} touches - {strength}")
    if count > 0 and count <= 5:
        print(f"  Dates: {[str(d.date()) for d in dates]}")

# Validate Resistance levels
print("\n" + "="*70)
print("RESISTANCE LEVELS VALIDATION:")
print("="*70)
for level in resistance_levels:
    count, dates = count_touches(data, level, tolerance_percent=2)
    strength = "STRONG 💪" if count >= 3 else "MODERATE ⚠️" if count == 2 else "WEAK ❓"
    print(f"\n${level}: {count} touches - {strength}")
    if count > 0 and count <= 5:
        print(f"  Dates: {[str(d.date()) for d in dates]}")

print("\n" + "="*70)